In [3]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        print(os.path.join(root, file))


/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [4]:
import pandas as pd

base_path = '/kaggle/input/competitions/ieee-fraud-detection/'

print("Loading data... (This might take a minute, the files are large)")

# 1. Load the training data
train_transaction = pd.read_csv(base_path + 'train_transaction.csv')
train_identity = pd.read_csv(base_path + 'train_identity.csv')

# Merge training data on 'TransactionID'
# We use how='left' because not every transaction has identity details
train_df = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
print(f"Train shape: {train_df.shape}")

# 2. Load the test data
test_transaction = pd.read_csv(base_path + 'test_transaction.csv')
test_identity = pd.read_csv(base_path + 'test_identity.csv')

# Merge test data on 'TransactionID'
test_df = pd.merge(test_transaction, test_identity, on='TransactionID', how='left')
print(f"Test shape: {test_df.shape}")

# Delete the original dataframes to free up RAM
del train_transaction, train_identity, test_transaction, test_identity

Loading data... (This might take a minute, the files are large)
Train shape: (590540, 434)
Test shape: (506691, 433)


In [5]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

def reduce_mem_usage(df):
    """Iterates through all columns and safely downcasts numeric types to save RAM."""
    start_mem = df.memory_usage().sum() / 1024**2
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            
            # Skip if the column is entirely NaN
            if pd.isna(c_min) or pd.isna(c_max):
                continue
                
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                # Cast numpy limits to standard float to prevent casting overflow warnings
                f16_min, f16_max = float(np.finfo(np.float16).min), float(np.finfo(np.float16).max)
                f32_min, f32_max = float(np.finfo(np.float32).min), float(np.finfo(np.float32).max)
                
                if c_min > f16_min and c_max < f16_max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > f32_min and c_max < f32_max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
                    
    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Optimized memory usage: {end_mem:.2f} MB')
    print(f'Decreased by {100 * (start_mem - end_mem) / start_mem:.1f}%\n')
    return df

print("Optimizing Train Data...")
train_df = reduce_mem_usage(train_df)
print("Optimizing Test Data...")
test_df = reduce_mem_usage(test_df)

Optimizing Train Data...
Optimized memory usage: 645.97 MB
Decreased by 67.0%

Optimizing Test Data...
Optimized memory usage: 561.50 MB
Decreased by 66.5%



In [6]:
#Preprocessing
import numpy as np
from sklearn.preprocessing import LabelEncoder

print("Handling missing values and encoding categoricals")

# Standardize column names in the test set to match the train set
test_df.columns = test_df.columns.str.replace('-', '_')

# Separate target and drop IDs
y_train = train_df['isFraud']
X_train = train_df.drop(['isFraud', 'TransactionID'], axis=1)
X_test = test_df.drop(['TransactionID'], axis=1)

# Ensure columns are exactly aligned, in same order
X_test = X_test[X_train.columns]

for col in X_train.columns:
    if X_train[col].dtype == 'object':
        X_train[col] = X_train[col].fillna('missing_value').astype(str)
        X_test[col] = X_test[col].fillna('missing_value').astype(str)
        
        le = LabelEncoder()
        # Fit on both train and test to capture all unique categories
        le.fit(list(X_train[col].values) + list(X_test[col].values))
        X_train[col] = le.transform(list(X_train[col].values))
        X_test[col] = le.transform(list(X_test[col].values))
    else:
        # Impute missing values with -999 for tree-based models
        X_train[col] = X_train[col].fillna(-999)
        X_test[col] = X_test[col].fillna(-999)

print("Data is strictly numeric and ready for modeling.")

Handling missing values and encoding categoricals
Data is strictly numeric and ready for modeling.


In [7]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, classification_report, roc_auc_score

print("Splitting data temporally (First 80% Train, Last 20% Validation)...")
# shuffle=False ensures we train on the past and predict on the future
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, shuffle=False)

# Calculate class weight to handle extreme imbalance
# scale_pos_weight = count(negative examples) / count(positive examples)
neg_cases = (y_tr == 0).sum()
pos_cases = (y_tr == 1).sum()
scale_weight = neg_cases / pos_cases
print(f"Calculated scale_pos_weight: {scale_weight:.2f}")

print("Initializing and training XGBoost Model...")
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_weight,
    objective='binary:logistic',
    eval_metric='aucpr', # Optimize directly for Precision-Recall Area Under Curve
    tree_method='hist',  # 'hist' is much faster for large datasets like IEEE
    random_state=42
)

# Train with early stopping to prevent overfitting
xgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=50
)

print("\nEvaluating Supervised Model...")
# Predict probabilities on the validation set
y_val_probs_xgb = xgb_model.predict_proba(X_val)[:, 1]

# Calculate PR-AUC 
pr_auc = average_precision_score(y_val, y_val_probs_xgb)
# Calculate ROC-AUC for comparison
roc_auc = roc_auc_score(y_val, y_val_probs_xgb)

print(f"ROC-AUC Score: {roc_auc:.4f}") #Often misleadingly high
print(f"PR-AUC Score:  {pr_auc:.4f}")  #The metric that matters

Splitting data temporally (First 80% Train, Last 20% Validation)...
Calculated scale_pos_weight: 27.46
Initializing and training XGBoost Model...
[0]	validation_0-aucpr:0.34065
[50]	validation_0-aucpr:0.44648
[100]	validation_0-aucpr:0.46903
[150]	validation_0-aucpr:0.48432
[200]	validation_0-aucpr:0.49205
[250]	validation_0-aucpr:0.50037
[299]	validation_0-aucpr:0.50601

Evaluating Supervised Model...
ROC-AUC Score: 0.9022
PR-AUC Score:  0.5061


In [8]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import MinMaxScaler
import numpy as np

print("Preparing data for the Neural Network...")
# 1. No negative imputations and unscaled data
X_tr_ae = X_tr.replace(-999, 0)
X_val_ae = X_val.replace(-999, 0)

# 2. Scale all features between 0 and 1
scaler_ae = MinMaxScaler()
X_tr_scaled = scaler_ae.fit_transform(X_tr_ae)
X_val_scaled = scaler_ae.transform(X_val_ae)

# 3. Train only on the majority that is legitimate class
X_tr_legit = X_tr_scaled[y_tr == 0]
print(f"Training Autoencoder on {X_tr_legit.shape[0]} legitimate transactions...")

# 4. Build the Autoencoder Architecture
input_dim = X_tr_legit.shape[1]
autoencoder = models.Sequential([
    layers.Dense(128, activation='relu', input_shape=(input_dim,)),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),  
    layers.Dense(128, activation='relu'),
    layers.Dense(input_dim, activation='linear') # Reconstruct original dimensions
])

autoencoder.compile(optimizer='adam', loss='mse')

# 5. Train the model using a large batch size for fast Kaggle execution
autoencoder.fit(
    X_tr_legit, X_tr_legit, 
    epochs=10, 
    batch_size=2048, 
    validation_split=0.1, 
    verbose=1
)

print("\nEvaluating Unsupervised Model...")
# 6. Reconstruct the validation set and calculate the Mean Squared Error 
val_reconstructions = autoencoder.predict(X_val_scaled, batch_size=2048)
val_mse = np.mean(np.power(X_val_scaled - val_reconstructions, 2), axis=1)

# The MSE acts as our anomaly probability score
pr_auc_ae = average_precision_score(y_val, val_mse)
roc_auc_ae = roc_auc_score(y_val, val_mse)

print(f"Autoencoder ROC-AUC: {roc_auc_ae:.4f}")
print(f"Autoencoder PR-AUC:  {pr_auc_ae:.4f}")

Preparing data for the Neural Network...
Training Autoencoder on 455833 legitimate transactions...


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-10-09 16:55:00.579837: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/10
201/201 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - loss: 0.0131 - val_loss: 0.0049
Epoch 2/10
201/201 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - loss: 0.0037 - val_loss: 0.0033
Epoch 3/10
201/201 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 0.0028 - val_loss: 0.0027
Epoch 4/10
201/201 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 0.0023 - val_loss: 0.0021
Epoch 5/10
201/201 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 0.0019 - val_loss: 0.0018
Epoch 6/10
201/201 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 0.0018 - val_loss: 0.0017
Epoch 7/10
201/201 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - loss: 0.0017 - val_loss: 0.0016
Epoch 8/10
201/201 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 0.0016 - val_loss: 0.0015
Epoch 9/10
201/201 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 0.0015 - val_loss: 0.0014
Epoch 10/10
201/201 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 0.0014 - val_loss: 0.0013

Evaluating Unsupervised Model...
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
Autoencoder ROC-AUC: 0.7107
Autoencoder PR-AUC:  0.0779


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

print("1. Normalizing Autoencoder MSE to 0-1 probabilities...")
# Fit scaler on the training MSE to prevent data leakage
train_reconstructions = autoencoder.predict(X_tr_scaled, batch_size=2048)
train_mse = np.mean(np.power(X_tr_scaled - train_reconstructions, 2), axis=1)

ae_scaler = MinMaxScaler()
ae_scaler.fit(train_mse.reshape(-1, 1))

# Transform Validation MSE
val_ae_prob = ae_scaler.transform(val_mse.reshape(-1, 1)).flatten()
val_ae_prob = np.clip(val_ae_prob, 0, 1) # Cap at 1.0

print("2. Training Meta-Learner (Stacking)...")
# Prepare the meta-features that is the outputs of Model A and Model B
# Use the validation set to train the meta-learner
X_meta_val = np.column_stack((y_val_probs_xgb, val_ae_prob))

# Train a fast Logistic Regression to learn how to weigh XGBoost vs Autoencoder
meta_model = LogisticRegression(class_weight='balanced')
meta_model.fit(X_meta_val, y_val)

# Get the final combined probabilities
final_ensemble_probs = meta_model.predict_proba(X_meta_val)[:, 1]
print(f"Meta-Learner PR-AUC: {average_precision_score(y_val, final_ensemble_probs):.4f}")

print("\n3. Cost-Sensitive Threshold Optimization...")
def optimize_threshold(y_true, y_probs, cost_fp=50, cost_fn=200):
    """
    Cost_FP: $50 to handle a blocked legitimate customer (support calls, friction).
    Cost_FN: $200 average loss for missing a fraudulent transaction.
    """
    thresholds = np.linspace(0.01, 0.99, 100)
    costs = []
    
    for thresh in thresholds:
        preds = (y_probs >= thresh).astype(int)
        
        # Calculate confusion matrix components
        false_positives = np.sum((preds == 1) & (y_true == 0))
        false_negatives = np.sum((preds == 0) & (y_true == 1))
        
        # Total Financial Cost
        total_cost = (false_positives * cost_fp) + (false_negatives * cost_fn)
        costs.append(total_cost)
        
    best_idx = np.argmin(costs)
    best_threshold = thresholds[best_idx]
    min_cost = costs[best_idx]
    
    return best_threshold, min_cost, thresholds, costs

optimal_thresh, min_cost, thresholds, costs = optimize_threshold(y_val, final_ensemble_probs)

print(f"Default 0.5 Threshold Cost: ${costs[49]:,.2f}")
print(f"Optimal Threshold: {optimal_thresh:.2f}")
print(f"Minimized Cost: ${min_cost:,.2f}")
print(f"Total Savings: ${(costs[49] - min_cost):,.2f}")

1. Normalizing Autoencoder MSE to 0-1 probabilities...
231/231 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step
2. Training Meta-Learner (Stacking)...
Meta-Learner PR-AUC: 0.5066

3. Cost-Sensitive Threshold Optimization...
Default 0.5 Threshold Cost: $973,650.00
Optimal Threshold: 0.91
Minimized Cost: $522,100.00
Total Savings: $451,550.00


In [10]:
import numpy as np
import pandas as pd

def calculate_psi(expected, actual, buckets=10):
    """
    Calculates the Population Stability Index (PSI).
    """
    # Create decile breakpoints based on the expected training data
    breakpoints = np.arange(0, buckets + 1) / buckets * 100
    expected_percents = np.percentile(expected, breakpoints)
    
    # Add slight perturbations to avoid boundary issues
    expected_percents[0] = expected_percents[0] - 0.0001
    expected_percents[-1] = expected_percents[-1] + 0.0001
    
    # Ensure all bin edges are unique (handles heavily skewed data with lots of 0s)
    expected_percents = np.unique(expected_percents)

    # Bin both datasets using the exact same edges
    expected_bins = pd.cut(expected, bins=expected_percents, include_lowest=True)
    actual_bins = pd.cut(actual, bins=expected_percents, include_lowest=True)
    
  
    expected_fractions = pd.Series(expected_bins).value_counts(sort=False).values / len(expected)
    actual_fractions = pd.Series(actual_bins).value_counts(sort=False).values / len(actual)
    
    # Replace zeros with a tiny number to prevent division-by-zero errors in the log function
    expected_fractions = np.where(expected_fractions == 0, 0.0001, expected_fractions)
    actual_fractions = np.where(actual_fractions == 0, 0.0001, actual_fractions)
    
    # PSI Formula: sum((Actual_frac - Expected_frac) * ln(Actual_frac / Expected_frac))
    psi_values = (actual_fractions - expected_fractions) * np.log(actual_fractions / expected_fractions)
    
    return np.sum(psi_values)

# --- Simulate the Daily Chron-Job ---

print("Checking for Data Drift...")


baseline_data = X_tr['V14'].head(50000).astype('float32').values
live_data = X_val['V14'].tail(50000).astype('float32').values

psi_score = calculate_psi(baseline_data, live_data)

print(f"Population Stability Index (PSI) for V14: {psi_score:.4f}")

# Industry Standard Alerting Logic
if psi_score < 0.1:
    print("Status: Stable. No significant drift detected.")
elif psi_score < 0.2:
    print("Status: Warning. Moderate covariate shift detected. Monitor closely.")
else:
    print("Status: ALERT! Significant drift detected. Auto-triggering retraining pipeline!")

Checking for Data Drift...
Population Stability Index (PSI) for V14: 0.2367
Status: ALERT! Significant drift detected. Auto-triggering retraining pipeline!


In [11]:
import numpy as np
from sklearn.metrics import average_precision_score

print("1. SIMULATING A ZERO-DAY FRAUD RING")
# We carve out 5,000 recent, purely legitimate transactions
X_live = X_val[y_val == 0].tail(5000).copy()
y_live = y_val[y_val == 0].tail(5000).copy()

# INJECT DRIFT: A new fraud syndicate attacks. 
# They exploit a new loophole, causing a massive artificial spike in V14 and a drop in V17.
attack_indices = X_live.sample(100, random_state=42).index
X_live.loc[attack_indices, 'V14'] = X_live['V14'].max() + 10  # Unprecedented spike
X_live.loc[attack_indices, 'V17'] = X_live['V17'].min() - 10  # Unprecedented drop
y_live.loc[attack_indices] = 1                                # Label them as Fraud

print(f"Injected 100 new fraud cases into a pool of {len(X_live)} transactions.")


stale_preds = xgb_model.predict_proba(X_live)[:, 1]
stale_pr_auc = average_precision_score(y_live, stale_preds)
print(f"Stale Model PR-AUC on New Attack: {stale_pr_auc:.4f} (Model is completely blind!)")


print("\n2. TRIGGERING ONLINE INCREMENTAL RETRAINING")


print("Updating model weights with the new data slice...")
xgb_model.fit(
    X_live, y_live, 
    xgb_model=xgb_model.get_booster(), 
    eval_set=[(X_live, y_live)],
    verbose=False
)

print("\n--- 3. EVALUATING THE PATCHED SYSTEM ---")

# Re-test the updated model on the same attack data
patched_preds = xgb_model.predict_proba(X_live)[:, 1]
patched_pr_auc = average_precision_score(y_live, patched_preds)

print(f"Patched Model PR-AUC on New Attack: {patched_pr_auc:.4f} (Threat neutralized!)")

1. SIMULATING A ZERO-DAY FRAUD RING
Injected 100 new fraud cases into a pool of 5000 transactions.
Stale Model PR-AUC on New Attack: 0.0189 (Model is completely blind!)

2. TRIGGERING ONLINE INCREMENTAL RETRAINING
Updating model weights with the new data slice...

--- 3. EVALUATING THE PATCHED SYSTEM ---
Patched Model PR-AUC on New Attack: 1.0000 (Threat neutralized!)
